# Realization-derived work: evidence review

## tl;dr

The model now prices proved physical K and only the scalar loops retained by each candidate. This notebook checks saved source, selection and numerical evidence; it does not launch GPU work. Model scores are not performance measurements. The broader PyTorch/MPS parity goal remains open.

## Context & Methods

Eight registered FP32 GEMM shapes, including four held-out shapes and 8192³. Each compiler chooses among the same fifteen blocks using only its model score; coefficients are unchanged. Fresh replay freezes both block and solved thread width.

### Key Assumptions

Native/Torch/direct-MPS use preallocated outputs and the same full FP64 oracle. GPU measurements are command-buffer intervals including work/gaps, not isolated kernel timestamps. Host batch/single E2E and GPU batch/single are separate. Concurrent desktop activity is not experimentally controlled; identical-source controls and per-round reversals must remain visible.

## Data

### 1. Load the registered experiment and independent auditor

Grain is shape × compiler × trial during selection, and shape × compiler × round during replay. This companion reads only the experiment subtree and its explicitly linked independent timing auditor.

In [1]:
import importlib.util
import json
from pathlib import Path
root = Path.cwd()
spec = importlib.util.spec_from_file_location('realized_work_audit', root / 'audit.py')
auditor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(auditor)
selection = json.loads((root / 'selection/results.json').read_text())
checked = auditor.selection(selection, root / 'selection')
print({k: v for k, v in checked.items() if k != 'summary'})

{'passed': True, 'attempted_candidates': 240, 'complete_outputs': 768, 'checked_elements': 12225970560, 'unchanged_fixed_mappings': 118, 'remapped_fixed_blocks': 2}


## Results

### 2. Exact model-selected mappings

Lookup by shape and variant; these are geometry and work counts, not timing comparisons.

In [2]:
for row in checked['summary']:
    plan = row['selected']
    print('x'.join(map(str, row['shape'])), row['variant'], 'heldout=', row['heldout'],
          'block=', plan['block'], 'threads=', plan['threads'],
          'physical/nominal=', (plan['physical_equivalent_issues'], plan['nominal_issues']),
          'retained scalar=', plan['retained_scalar_elements'])
assert checked['complete_outputs'] == 768
assert checked['unchanged_fixed_mappings'] + checked['remapped_fixed_blocks'] == 120

512x512x512 reference heldout= False block= [128, 64, 512] threads= 512 physical/nominal= (8192, 8192.0) retained scalar= 24576
512x512x512 candidate heldout= False block= [128, 64, 512] threads= 512 physical/nominal= (8192, 8192.0) retained scalar= 0
4096x4096x4096 candidate heldout= False block= [128, 64, 4096] threads= 256 physical/nominal= (65536, 65536.0) retained scalar= 0
4096x4096x4096 reference heldout= False block= [128, 64, 4096] threads= 256 physical/nominal= (65536, 65536.0) retained scalar= 24576
1025x1025x1024 reference heldout= False block= [32, 64, 512] threads= 64 physical/nominal= (4096, 4096.0) retained scalar= 8192
1025x1025x1024 candidate heldout= False block= [64, 64, 4096] threads= 128 physical/nominal= (8192, 32768.0) retained scalar= 0
4096x4096x11008 candidate heldout= False block= [128, 64, 4096] threads= 256 physical/nominal= (176128, 196608.0) retained scalar= 0
4096x4096x11008 reference heldout= False block= [128, 64, 512] threads= 256 physical/nominal= (

### 3. Full-build correctness and retained failed attempts

Nonzero assertions are mandatory. The first failures were a test-oracle prior omission and an unrecognized equivalent bound expression, not numerical output failures. Their receipts remain separate from final passing evidence.

In [3]:
receipt = json.loads((root / 'final-correctness/results.json').read_text())
auditor.gate(receipt, 'check')
assert len(receipt['results']) == 12
assert all(row['passed'] and row['passed_assertions'] > 0 for row in receipt['results'])
print('Passing assertions:', sum(row['passed_assertions'] for row in receipt['results']))
for tag in ('correctness', 'normalization-diagnostic'):
    failed = json.loads((root / tag / 'results.json').read_text())
    assert failed['passed'] is False
    print(tag, [row['log'] for row in failed['results'] if not row['passed']])

Passing assertions: 915617
correctness ['test_tile_tirx_planner-unit.log', 'test_tile_tirx_matrix-metal.log']
normalization-diagnostic ['test_tile_tirx_matrix-metal.log']


### 4. Six-round frozen GPU/E2E replay

Ratios are candidate time divided by reference/provider time, lower is better. Medians of paired round ratios are not confidence intervals. Identical sources cannot be credited as compiler speedups.

In [4]:
replay = json.loads((root / 'replay/results.json').read_text())
paired = auditor.replay(replay, root / 'replay')
print({k: v for k, v in paired.items() if k != 'summary'})
for row in paired['summary']:
    same_source = row['mappings']['reference']['source'] == row['mappings']['candidate']['source']
    gpu, host = (row['metrics'][key] for key in ('gpu_throughput', 'e2e_throughput'))
    print('x'.join(map(str, row['shape'])), 'same source=', same_source,
          'GPU new/old=', gpu['new_old'], 'E2E new/old=', host['new_old']['median'],
          'GPU new/Torch=', gpu['new_torch']['median'], 'GPU new/MPS=', gpu['new_mps']['median'])

{'passed': True, 'complete_outputs': 288, 'checked_elements': 4584738960, 'paired_rounds': 48}
512x512x512 same source= True GPU new/old= {'median': 1.0046652757069445, 'minimum': 0.9948637700347327, 'maximum': 1.0153827775360529, 'faster_rounds': 2, 'ratios': [1.0003384048992285, 0.9948637700347327, 1.0089921465146603, 1.0106349889905388, 1.0153827775360529, 0.9983467260511643]} E2E new/old= 1.0033908491337824 GPU new/Torch= 1.1657451657775026 GPU new/MPS= 0.9661487484843032
4096x4096x4096 same source= True GPU new/old= {'median': 0.9927420236101876, 'minimum': 0.9876887362047706, 'maximum': 1.008445453330308, 'faster_rounds': 5, 'ratios': [0.997716591377585, 1.008445453330308, 0.9879618174004168, 0.9932161547043868, 0.9876887362047706, 0.9922678925159885]} E2E new/old= 1.0041602406696755 GPU new/Torch= 1.269505312063735 GPU new/MPS= 1.162117144239394
1025x1025x1024 same source= False GPU new/old= {'median': 0.9612400923184008, 'minimum': 0.9444595470385406, 'maximum': 0.9696896695605

## Takeaways

The proof/realization contract is reusable within the admitted matrix family; unsupported bounds retain conservative nominal work. The finite replay cannot establish universal speed or model calibration. Check same-source controls, all paired reversals and allocation/timing definitions before interpreting a changed schedule. Native Metal, XIR/SIMD and non-matrix operators do not inherit a performance claim from this matrix-model correction. Reader-facing conclusions and limitations live in the existing Sphinx route-results page.